In [1]:
import pyspark

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
        .master('local[2]') \
        .appName('05_taxi_schema') \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/21 13:30:13 WARN Utils: Your hostname, Prajwals-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.9.200.126 instead (on interface en0)
26/02/21 13:30:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/21 13:30:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
import pandas as pd

In [5]:
# Create the directory
!mkdir -p data/raw/yellow/2025/11

In [6]:
# Download Yellow Taxi November 2025
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet -O data/raw/yellow/2025/11/data.parquet

--2026-02-21 13:30:20--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.161.140.125, 18.161.140.124, 18.161.140.41, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.161.140.125|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘data/raw/yellow/2025/11/data.parquet’

data/raw/yellow/202 100%[===================>]  67.84M  17.5MB/s    in 3.9s    

2026-02-21 13:30:24 (17.4 MB/s) - ‘data/raw/yellow/2025/11/data.parquet’ saved [71134255/71134255]



In [7]:
# Read without a schema just to see what's inside
temp_df = spark.read.parquet("data/raw/yellow/2025/11/data.parquet")
temp_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [8]:
from pyspark.sql import types

yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampNTZType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampNTZType(), True),
    types.StructField("passenger_count", types.LongType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.LongType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.LongType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True),
    types.StructField("Airport_fee", types.DoubleType(), True),
    types.StructField("cbd_congestion_fee", types.DoubleType(), True)
])

In [9]:
year = 2025
month = 11

# If you only have November, you can use [11]. For a full year, use range(1, 13)
for month in [11]:
    print(f'processing data for {year}/{month:02d} ...')

    input_path = f'data/raw/yellow/{year}/{month:02d}/'
    output_path = f'data/pq/yellow/{year}/{month:02d}/'

    # 1. Read the data
    # Note: Using .schema() with Parquet is optional but recommended for strictness
    df_yellow = spark.read \
                .schema(yellow_schema) \
                .parquet(input_path)

    # 2. Perform Transformations (Optional: e.g., adding a processing date)
    # df_yellow = df_yellow.withColumn('processed_at', F.current_timestamp())


    # 3. Repartition and Save
    # Repartition(4) ensures that even if the input was 1 giant file, 
    # we save it as 4 smaller, manageable files.

    df_yellow.repartition(4) \
            .write \
            .mode('overwrite') \
            .parquet(output_path) 


    print(f'successfully wrote repartitioned df to: {output_path} ..')
    

processing data for 2025/11 ...


[Stage 3:=============================>                             (2 + 2) / 4]

successfully wrote repartitioned df to: data/pq/yellow/2025/11/ ..


In [10]:
!ls -lh data/pq/yellow/2025/11/

total 213504
-rw-r--r--@ 1 prajwalchambenandeeshappa  staff     0B Feb 21 13:30 _SUCCESS
-rw-r--r--@ 1 prajwalchambenandeeshappa  staff    25M Feb 21 13:30 part-00000-e4e9f14c-b0e3-4e3a-84bc-073fdbee4ed6-c000.snappy.parquet
-rw-r--r--@ 1 prajwalchambenandeeshappa  staff    25M Feb 21 13:30 part-00001-e4e9f14c-b0e3-4e3a-84bc-073fdbee4ed6-c000.snappy.parquet
-rw-r--r--@ 1 prajwalchambenandeeshappa  staff    25M Feb 21 13:30 part-00002-e4e9f14c-b0e3-4e3a-84bc-073fdbee4ed6-c000.snappy.parquet
-rw-r--r--@ 1 prajwalchambenandeeshappa  staff    25M Feb 21 13:30 part-00003-e4e9f14c-b0e3-4e3a-84bc-073fdbee4ed6-c000.snappy.parquet


In [11]:
# Finding File size for Yellow November 2025 parquet file = 68M
!ls -lh data/raw/yellow/2025/11/data.parquet


-rw-r--r--@ 1 prajwalchambenandeeshappa  staff    68M Dec 19 09:51 data/raw/yellow/2025/11/data.parquet


In [12]:
#total records count for yellow taxi dataset for november 2025

#read newly created parquet file
df_processed = spark.read.parquet('data/pq/yellow/2025/11/')

#count the rows
df_count = df_processed.count()

print(f'total records count for yellow taxi dataset for november 2025 is {df_count}')

total records count for yellow taxi dataset for november 2025 is 4181444


In [13]:
#create directory for green taxi nov 2025
!mkdir -p data/raw/green/2025/11/

In [14]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-11.parquet -O data/raw/green/2025/11/data.parquet

--2026-02-21 13:30:41--  https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.161.140.125, 18.161.140.124, 18.161.140.41, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.161.140.125|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1164775 (1.1M) [binary/octet-stream]
Saving to: ‘data/raw/green/2025/11/data.parquet’

data/raw/green/2025 100%[===================>]   1.11M  --.-KB/s    in 0.07s   

2026-02-21 13:30:42 (17.0 MB/s) - ‘data/raw/green/2025/11/data.parquet’ saved [1164775/1164775]



In [15]:
#total records count for green taxi dataset for november 2025

#read newly created parquet file
df_green = spark.read.parquet('data/raw/green/2025/11/')

#count the rows
df_count = df_green.count()

print(f'total records count for green taxi dataset for november 2025 is {df_count}')

total records count for green taxi dataset for november 2025 is 46912


In [16]:
df_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- trip_type: long (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [18]:
from pyspark.sql import functions as F

# 1. Calculate the duration in seconds
# We cast TIMESTAMP_NTZ -> TIMESTAMP -> LONG
df_green = df_green.withColumn('duration_seconds', 
    F.col('lpep_dropoff_datetime').cast('timestamp').cast('long') - 
    F.col('lpep_pickup_datetime').cast('timestamp').cast('long')
)

# 2. Convert seconds to hours
df_green = df_green.withColumn('duration_hours', F.col('duration_seconds') / 3600)

# 3. Find the maximum value
longest_trip = df_green.select(F.max('duration_hours')).collect()[0][0]

print(f"The longest trip duration is: {longest_trip:.1f} hours")

The longest trip duration is: 24.8 hours


In [25]:
from pyspark.sql import functions as F

# 1. Calculate the duration in seconds
# We cast TIMESTAMP_NTZ -> TIMESTAMP -> LONG
df_yellow = df_yellow.withColumn('duration_seconds', 
    F.col('tpep_dropoff_datetime').cast('timestamp').cast('long') - 
    F.col('tpep_pickup_datetime').cast('timestamp').cast('long')
)

print(df_yellow['duration_seconds'])

# 2. Convert seconds to hours
df_yellow = df_yellow.withColumn('duration_hours', F.col('duration_seconds') / 3600)

# 3. Find the maximum value
longest_trip = df_yellow.select(F.max('duration_hours')).collect()[0][0]

print(f"The longest trip duration is: {longest_trip:.1f} hours")

Column<'duration_seconds'>
The longest trip duration is: 90.6 hours


In [35]:
from pyspark.sql import functions as F

df_zones = spark.read.option('header','true').csv('taxi+_zone_lookup.csv')
df_zones.show()

df_PULocationID_count = df_yellow.groupBy('PULocationID').count()
df_PULocationID_count.show()

df_zone_pickup_loc = df_PULocationID_count.join(df_zones, df_PULocationID_count.PULocationID == df_zones.LocationID)


df_zone_pickup_loc.select('zone','count').orderBy('count', ascending=True).show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly